# 🛒 Segmentation Clientèle E-Commerce — Pipeline Big Data Complet
## Online Retail Dataset — UK E-Commerce (2010-2011)

Ce notebook implémente un pipeline complet de segmentation client en **deux approches** :
- 🔵 **PySpark** : traitement Big Data distribué (production-ready)
- 🟢 **Pandas + Scikit-learn** : analyse complète avec visualisations avancées

**Objectif** : Identifier des segments clients actionnables via la méthode **RFM** (Recency, Frequency, Monetary) et construire un modèle prédictif pour anticiper les clients à forte valeur.

---
| Étape | Description |
|-------|-------------|
| 1️⃣ | Chargement & exploration des données |
| 2️⃣ | Nettoyage & Feature Engineering RFM |
| 3️⃣ | Segmentation non-supervisée (K-Means) |
| 4️⃣ | Profilage & visualisations des segments |
| 5️⃣ | Modèle prédictif supervisé (Random Forest) |
| 6️⃣ | Optimisation (GridSearchCV) & évaluation finale |


---
## 1️⃣ Chargement & Exploration des Données


### 📦 1.1 Imports et configuration

On importe toutes les bibliothèques nécessaires pour les deux approches (PySpark et Scikit-learn).


In [ ]:
# ── Bibliothèques générales ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import FancyBboxPatch
from matplotlib.gridspec import GridSpec

# ── Machine Learning (Scikit-learn) ───────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (silhouette_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                      cross_val_score, StratifiedKFold)

# ── Style global ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

PALETTE = ['#0E7C7B', '#0D1B2A', '#F59E0B', '#EF4444']  # teal, navy, gold, red
print("✅ Imports effectués avec succès")


### 🔵 1.2 Approche PySpark — Initialisation et chargement

> **Note** : Cette section présente le code PySpark pour un environnement de production.
> Exécutez-la si PySpark est installé (`pip install pyspark`).


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 🔵 APPROCHE PYSPARK — Production Big Data
# ════════════════════════════════════════════════════════════════════════

# Pour exécuter, décommenter les lignes ci-dessous :

# from pyspark.sql import SparkSession, functions as F
# from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler
# from pyspark.ml.clustering import KMeans as SparkKMeans
# from pyspark.ml.evaluation import ClusteringEvaluator
# from pyspark.ml.classification import RandomForestClassifier as SparkRF
# from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
# from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# spark = SparkSession.builder \
#     .appName("Segmentation_Retail") \
#     .config("spark.driver.memory", "4g") \
#     .getOrCreate()

# spark.sparkContext.setLogLevel("WARN")

# DATASET_PATH = "file:///chemin/vers/Online_Retail_CSV.csv"
# df_spark = spark.read.csv(DATASET_PATH, header=True, inferSchema=True, sep=";")

# df_spark.printSchema()
# print(f"Nombre de lignes : {df_spark.count():,}")
# df_spark.show(5)

print("ℹ️  Section PySpark prête — décommentez pour exécuter sur votre cluster Spark")


### 🟢 1.3 Chargement Pandas et exploration initiale

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 🟢 APPROCHE PANDAS — Analyse locale complète
# ════════════════════════════════════════════════════════════════════════

DATASET_PATH = "Online_Retail_CSV.csv"   # ← Adaptez ce chemin si nécessaire

df = pd.read_csv(DATASET_PATH, sep=';', encoding='utf-8-sig')

print(f"📊 Dimensions du dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"📅 Période couverte : à déterminer après conversion des dates")
print("\n🔍 Aperçu des premières lignes :")
df.head()


In [ ]:
# ── Statistiques générales ────────────────────────────────────────────────
print("=" * 55)
print("  INFORMATIONS GÉNÉRALES DU DATASET")
print("=" * 55)
print(f"  {'Colonnes':<25} : {df.shape[1]}")
print(f"  {'Lignes totales':<25} : {df.shape[0]:,}")
print(f"  {'Clients uniques':<25} : {df['CustomerID'].nunique():,}")
print(f"  {'Pays représentés':<25} : {df['Country'].nunique()}")
print(f"  {'Produits distincts':<25} : {df['StockCode'].nunique():,}")
print("=" * 55)

print("\n📌 Valeurs manquantes :")
missing = df.isnull().sum()
print(missing[missing > 0].to_string())


---
## 2️⃣ Nettoyage & Feature Engineering RFM


### 🔵 2.1 PySpark — Prétraitement et construction RFM

Le code PySpark ci-dessous réalise le même pipeline que l'approche Pandas, mais distribué.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 🔵 PYSPARK — Prétraitement & RFM
# ════════════════════════════════════════════════════════════════════════

# Décommenter pour exécuter :

# # 1. Correction du type UnitPrice (virgule → point)
# df_spark = df_spark.withColumn("UnitPrice",
#     F.regexp_replace(F.col("UnitPrice"), ",", ".").cast("float"))

# # 2. Calcul du total par ligne
# df_spark = df_spark.withColumn("TotalPrice",
#     F.col("Quantity") * F.col("UnitPrice"))

# # 3. Nettoyage : suppression des lignes invalides
# df_clean_spark = df_spark \
#     .filter(F.col("CustomerID").isNotNull()) \
#     .filter(F.col("Quantity") > 0) \
#     .filter(F.col("UnitPrice") > 0) \
#     .withColumn("InvoiceDate",
#         F.to_timestamp(F.col("InvoiceDate"), "dd/MM/yyyy HH:mm"))

# # 4. Construction de la table RFM
# ref_date_spark = df_clean_spark.select(F.max("InvoiceDate")).collect()[0][0]
# rfm_spark = df_clean_spark.groupBy("CustomerID").agg(
#     F.datediff(F.lit(ref_date_spark), F.max("InvoiceDate")).alias("Recency"),
#     F.countDistinct("InvoiceNo").alias("Frequency"),
#     F.round(F.sum("TotalPrice"), 2).alias("Monetary")
# )

# # 5. Filtrage des outliers (99e percentile)
# quant = rfm_spark.stat.approxQuantile(["Frequency", "Monetary"], [0.99], 0.01)
# rfm_spark = rfm_spark \
#     .filter(F.col("Frequency") <= quant[0][0]) \
#     .filter(F.col("Monetary") <= quant[1][0])

# rfm_spark.describe().show()

print("ℹ️  Code PySpark prêt — décommentez pour exécuter")


### 🟢 2.2 Pandas — Nettoyage des données

In [ ]:
# ── 1. Conversion des types ───────────────────────────────────────────────
df['UnitPrice']  = df['UnitPrice'].str.replace(',', '.').astype(float)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%d/%m/%Y %H:%M')
df['CustomerID']  = df['CustomerID'].astype('Int64')

# ── 2. Filtrage qualité ───────────────────────────────────────────────────
df_clean = df.dropna(subset=['CustomerID']).copy()
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# ── 3. Feature : Total par ligne ─────────────────────────────────────────
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(f"✅ Après nettoyage : {df_clean.shape[0]:,} lignes "
      f"({df_clean.shape[0]/df.shape[0]*100:.1f}% du total)")
print(f"   Lignes supprimées : {df.shape[0] - df_clean.shape[0]:,}")
print(f"   Période : {df_clean['InvoiceDate'].min().date()} → {df_clean['InvoiceDate'].max().date()}")


### 🟢 2.3 Construction de la table RFM

La méthode **RFM** transforme un historique de transactions en 3 indicateurs par client :

| Indicateur | Définition | Interprétation |
|------------|-----------|----------------|
| **Recency** | Jours depuis le dernier achat | Plus faible = plus actif |
| **Frequency** | Nombre de commandes distinctes | Plus élevé = plus fidèle |
| **Monetary** | Montant total dépensé | Plus élevé = plus rentable |


In [ ]:
# ── Date de référence : lendemain de la dernière transaction ─────────────
ref_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"📅 Date de référence : {ref_date.date()}")

# ── Agrégation RFM ────────────────────────────────────────────────────────
rfm = df_clean.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate',  lambda x: (ref_date - x.max()).days),
    Frequency = ('InvoiceNo',    'nunique'),
    Monetary  = ('TotalPrice',   'sum')
).reset_index()

rfm['Monetary'] = rfm['Monetary'].round(2)

print(f"\n📊 Table RFM : {rfm.shape[0]:,} clients")
print("\n📈 Statistiques descriptives :")
rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2)


In [ ]:
# ── Gestion des outliers (99e percentile) ────────────────────────────────
for col in ['Frequency', 'Monetary']:
    seuil = rfm[col].quantile(0.99)
    avant = len(rfm)
    rfm = rfm[rfm[col] <= seuil]
    print(f"  [{col}] Seuil p99 = {seuil:.1f} | "
          f"Clients retirés : {avant - len(rfm)}")

print(f"\n✅ Table RFM finale : {len(rfm):,} clients")


### 🟢 2.4 Visualisation des distributions RFM

Avant de clustériser, on visualise les distributions pour comprendre l'asymétrie des données.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Distributions RFM — Brut vs Transformé (log1p)", fontsize=15, fontweight='bold', y=1.01)

rfm_cols = ['Recency', 'Frequency', 'Monetary']
rfm_log  = np.log1p(rfm[rfm_cols])

for i, col in enumerate(rfm_cols):
    # Distribution brute
    ax = axes[0, i]
    ax.hist(rfm[col], bins=50, color=PALETTE[i], edgecolor='white', alpha=0.85)
    ax.set_title(f"{col} — Brut", fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel("Nb clients" if i == 0 else "")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f"{x:,.0f}"))

    # Distribution log-transformée
    ax2 = axes[1, i]
    ax2.hist(rfm_log[col], bins=50, color=PALETTE[i], edgecolor='white', alpha=0.85)
    ax2.set_title(f"log1p({col}) — Normalisé", fontweight='bold')
    ax2.set_xlabel(f"log1p({col})")

plt.tight_layout()
plt.show()
print("💡 La transformation log1p réduit l'asymétrie — indispensable avant le clustering.")


---
## 3️⃣ Segmentation Non-Supervisée — K-Means

### 🔵 3.1 PySpark — Clustering avec BisectingKMeans


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 🔵 PYSPARK — BisectingKMeans + Silhouette
# ════════════════════════════════════════════════════════════════════════

# from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkScaler
# from pyspark.ml.clustering import BisectingKMeans
# from pyspark.ml.evaluation import ClusteringEvaluator

# # Log-transformation
# for col in ["Recency", "Frequency", "Monetary"]:
#     rfm_spark = rfm_spark.withColumn(f"{col}_log", F.log1p(F.col(col)))

# # Assemblage + Standardisation
# assembler = VectorAssembler(
#     inputCols=["Recency_log", "Frequency_log", "Monetary_log"],
#     outputCol="features")
# scaler = SparkScaler(inputCol="features", outputCol="scaledFeatures",
#                      withStd=True, withMean=True)

# rfm_vec    = assembler.transform(rfm_spark)
# rfm_scaled_spark = scaler.fit(rfm_vec).transform(rfm_vec)

# # Recherche du meilleur k via Silhouette
# evaluator = ClusteringEvaluator(featuresCol="scaledFeatures",
#                                  metricName="silhouette")
# results = {}
# for k in range(2, 7):
#     bkm = BisectingKMeans(featuresCol="scaledFeatures", k=k,
#                           seed=42, maxIter=50, minDivisibleClusterSize=0.05)
#     model = bkm.fit(rfm_scaled_spark)
#     preds = model.transform(rfm_scaled_spark)
#     score = evaluator.evaluate(preds)
#     results[k] = score
#     print(f"k={k} | silhouette={score:.4f}")

# best_k_spark = max(results, key=results.get)
# print(f"\n🏆 Meilleur k Spark : {best_k_spark}")

print("ℹ️  Code PySpark prêt — décommentez pour exécuter")


### 🟢 3.2 Pandas — Sélection du nombre de clusters (Elbow + Silhouette)

In [ ]:
# ── Préparation des features ──────────────────────────────────────────────
rfm_log_vals = np.log1p(rfm[['Recency', 'Frequency', 'Monetary']])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_log_vals)

# ── Méthodes Elbow + Silhouette ───────────────────────────────────────────
inertias, silhouettes, k_range = [], [], range(2, 9)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=15, max_iter=300)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    print(f"  k={k} | Inertie={km.inertia_:>10.0f} | Silhouette={silhouette_score(X_scaled, labels):.4f}")

print(f"\n🏆 Silhouette maximale à k={k_range.start + np.argmax(silhouettes)}")


In [ ]:
# ── Graphique Elbow + Silhouette ──────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sélection du nombre de clusters optimal", fontsize=14, fontweight='bold')

k_list = list(k_range)

# Elbow
ax1.plot(k_list, inertias, 'o-', color=PALETTE[0], linewidth=2.5, markersize=8)
ax1.fill_between(k_list, inertias, alpha=0.1, color=PALETTE[0])
ax1.set_title("Méthode du Coude (Elbow)", fontweight='bold')
ax1.set_xlabel("Nombre de clusters k")
ax1.set_ylabel("Inertie (Within-cluster SSE)")
ax1.axvline(x=4, color=PALETTE[3], linestyle='--', alpha=0.7, label='k=4 choisi')
ax1.legend()

# Silhouette
colors_sil = [PALETTE[3] if s == max(silhouettes) else PALETTE[1] for s in silhouettes]
bars = ax2.bar(k_list, silhouettes, color=colors_sil, edgecolor='white', width=0.6)
ax2.set_title("Score Silhouette par k", fontweight='bold')
ax2.set_xlabel("Nombre de clusters k")
ax2.set_ylabel("Score Silhouette (↑ meilleur)")
for bar, val in zip(bars, silhouettes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


### 🟢 3.3 Entraînement du modèle K-Means final (k=4)

Nous choisissons **k=4** pour obtenir 4 segments marketing distincts et actionnables.


In [ ]:
# ── Entraînement final ────────────────────────────────────────────────────
BEST_K = 4
km_final = KMeans(n_clusters=BEST_K, random_state=42, n_init=15, max_iter=300)
rfm['Cluster'] = km_final.fit_predict(X_scaled)

# ── Profil de chaque cluster ──────────────────────────────────────────────
profil = rfm.groupby('Cluster').agg(
    Recency_moy   = ('Recency',    'mean'),
    Frequency_moy = ('Frequency',  'mean'),
    Monetary_moy  = ('Monetary',   'mean'),
    Nb_clients    = ('CustomerID', 'count')
).round(1).sort_values('Monetary_moy', ascending=False).reset_index()

print("\n📊 Profil des 4 clusters :\n")
print(profil.to_string(index=False))


### 🟢 3.4 Attribution des noms métier aux segments

On associe un nom business à chaque cluster selon son profil RFM.


In [ ]:
# ── Mapping cluster → nom métier ─────────────────────────────────────────
# Tri par Monetary_moy décroissant pour attribution logique
segment_names = {
    profil.iloc[0]['Cluster']: ('Champions / VIP',        '⭐', PALETTE[0]),
    profil.iloc[1]['Cluster']: ('Fidèles Réguliers',       '💚', '#22C55E'),
    profil.iloc[2]['Cluster']: ('Clients à Potentiel',     '🌱', PALETTE[2]),
    profil.iloc[3]['Cluster']: ('À Risque / Inactifs',     '⚠️',  PALETTE[3]),
}

rfm['Segment']    = rfm['Cluster'].map(lambda c: segment_names[c][0])
rfm['SegIcon']    = rfm['Cluster'].map(lambda c: segment_names[c][1])
rfm['SegColor']   = rfm['Cluster'].map(lambda c: segment_names[c][2])

print("✅ Segments attribués :\n")
print(rfm['Segment'].value_counts().to_string())


---
## 4️⃣ Visualisations des Segments


In [ ]:
# ── Tableau récapitulatif des segments ───────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
ax.axis('off')
fig.suptitle("📊 Profil des 4 Segments Clients", fontsize=15, fontweight='bold', y=1.02)

seg_summary = rfm.groupby('Segment').agg(
    Nb_clients    = ('CustomerID', 'count'),
    Recency_moy   = ('Recency',   'mean'),
    Frequency_moy = ('Frequency', 'mean'),
    Monetary_moy  = ('Monetary',  'mean'),
).round(1)

seg_summary['Part (%)'] = (seg_summary['Nb_clients'] / len(rfm) * 100).round(1)
seg_summary = seg_summary.sort_values('Monetary_moy', ascending=False)

table_data = []
headers = ['Segment', 'Nb Clients', 'Part (%)', 'Récence moy.', 'Fréq. moy.', 'Montant moy.']
for seg, row in seg_summary.iterrows():
    icon = rfm[rfm['Segment']==seg]['SegIcon'].iloc[0]
    table_data.append([
        f"{icon} {seg}",
        f"{int(row['Nb_clients']):,}",
        f"{row['Part (%)']:.1f}%",
        f"{row['Recency_moy']:.0f} j",
        f"{row['Frequency_moy']:.1f}",
        f"{row['Monetary_moy']:,.0f} €",
    ])

tbl = ax.table(cellText=table_data, colLabels=headers,
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 2.2)

colors_header = [PALETTE[1]] * len(headers)
for j, c in enumerate(colors_header):
    tbl[0, j].set_facecolor(c)
    tbl[0, j].set_text_props(color='white', fontweight='bold')

row_colors = [PALETTE[0], '#22C55E', PALETTE[2], PALETTE[3]]
for i, color in enumerate(row_colors):
    for j in range(len(headers)):
        tbl[i+1, j].set_facecolor(color + '22')

plt.tight_layout()
plt.show()


In [ ]:
# ── Scatter plot 3D → 2D RFM ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("🗺️  Cartographie RFM des Segments Clients", fontsize=15, fontweight='bold')

pairs = [
    ('Recency',   'Monetary',   axes[0], 'Récence vs Montant'),
    ('Frequency', 'Monetary',   axes[1], 'Fréquence vs Montant'),
    ('Recency',   'Frequency',  axes[2], 'Récence vs Fréquence'),
]

for xcol, ycol, ax, title in pairs:
    for seg, grp in rfm.groupby('Segment'):
        color = grp['SegColor'].iloc[0]
        icon  = grp['SegIcon'].iloc[0]
        ax.scatter(grp[xcol], grp[ycol], c=color, label=f"{icon} {seg}",
                   alpha=0.5, s=18, edgecolors='none')
    ax.set_xlabel(xcol, fontweight='bold')
    ax.set_ylabel(ycol, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    if ycol == 'Monetary':
        ax.set_yscale('log')
        ax.set_ylabel(f"{ycol} (log)", fontweight='bold')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=10,
           frameon=True, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.show()


In [ ]:
# ── Boxplots RFM par segment ──────────────────────────────────────────────
seg_order = rfm.groupby('Segment')['Monetary'].mean().sort_values(ascending=False).index.tolist()
seg_palette = {s: rfm[rfm['Segment']==s]['SegColor'].iloc[0] for s in rfm['Segment'].unique()}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("📦 Distribution RFM par Segment", fontsize=15, fontweight='bold')

for i, (col, ax) in enumerate(zip(['Recency','Frequency','Monetary'], axes)):
    sns.boxplot(data=rfm, x='Segment', y=col, order=seg_order,
                palette=seg_palette, ax=ax, showfliers=False, width=0.5,
                linewidth=1.5)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    if col == 'Monetary':
        ax.set_yscale('log')

plt.tight_layout()
plt.show()


In [ ]:
# ── Donut chart — Répartition des segments ───────────────────────────────
seg_counts = rfm['Segment'].value_counts()
seg_colors = [rfm[rfm['Segment']==s]['SegColor'].iloc[0] for s in seg_counts.index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("🎯 Répartition de la Base Clients", fontsize=15, fontweight='bold')

# Donut
wedges, texts, autotexts = ax1.pie(
    seg_counts.values, labels=None,
    colors=seg_colors, autopct='%1.1f%%',
    startangle=90, pctdistance=0.82,
    wedgeprops={'width': 0.55, 'edgecolor': 'white', 'linewidth': 3}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
    at.set_color('white')

ax1.set_title("Répartition en %", fontweight='bold')
ax1.legend([f"{rfm[rfm['Segment']==s]['SegIcon'].iloc[0]} {s}" for s in seg_counts.index],
           loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=9)

# Bar chart valeur totale par segment
seg_value = rfm.groupby('Segment')['Monetary'].sum().reindex(seg_counts.index)
bars = ax2.barh(range(len(seg_value)), seg_value.values/1e6,
                color=seg_colors, edgecolor='white', height=0.55)
ax2.set_yticks(range(len(seg_value)))
ax2.set_yticklabels([f"{rfm[rfm['Segment']==s]['SegIcon'].iloc[0]} {s}" for s in seg_value.index])
ax2.set_xlabel("Valeur totale (M€)")
ax2.set_title("Valeur Totale Générée par Segment (M€)", fontweight='bold')
for bar, val in zip(bars, seg_value.values):
    ax2.text(val/1e6 + 0.05, bar.get_y() + bar.get_height()/2,
             f"{val/1e6:.1f}M€", va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()


---
## 5️⃣ Modèle Prédictif Supervisé — Random Forest

**Objectif** : Prédire si un client sera un **"Gros Dépensier"** (Monetary > 1 000 €) à partir de sa Récence et Fréquence.

### 🔵 5.1 PySpark — Random Forest Classifier


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 🔵 PYSPARK — RandomForestClassifier + CrossValidator
# ════════════════════════════════════════════════════════════════════════

# from pyspark.ml.classification import RandomForestClassifier as SparkRF
# from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
# from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# # Création du label
# rfm_supervised = rfm_spark.withColumn("label",
#     F.when(F.col("Monetary") > 1000, 1).otherwise(0))

# # Features
# assembler_sup = VectorAssembler(
#     inputCols=["Recency", "Frequency"], outputCol="features")
# data_sup = assembler_sup.transform(rfm_supervised)
# train_df, test_df = data_sup.randomSplit([0.7, 0.3], seed=42)

# # Modèle de base
# rf_spark = SparkRF(labelCol="label", featuresCol="features",
#                    numTrees=100, seed=42)
# rf_model_spark = rf_spark.fit(train_df)
# preds_spark = rf_model_spark.transform(test_df)

# # Évaluation
# eval_spark = MulticlassClassificationEvaluator(labelCol="label",
#                                                 predictionCol="prediction")
# print("Accuracy :", eval_spark.evaluate(preds_spark, {eval_spark.metricName: "accuracy"}))
# print("F1       :", eval_spark.evaluate(preds_spark, {eval_spark.metricName: "f1"}))

# # Optimisation GridSearch + CrossValidation
# paramGrid = (ParamGridBuilder()
#     .addGrid(rf_spark.maxDepth, [5, 10])
#     .addGrid(rf_spark.numTrees, [50, 100, 200])
#     .build())

# cv = CrossValidator(estimator=rf_spark, estimatorParamMaps=paramGrid,
#                     evaluator=eval_spark, numFolds=5, seed=42)
# cv_model = cv.fit(train_df)
# print("Best maxDepth :", cv_model.bestModel.depth)
# print("Best numTrees :", cv_model.bestModel.getNumTrees)

print("ℹ️  Code PySpark prêt — décommentez pour exécuter")


### 🟢 5.2 Pandas — Préparation du modèle

In [ ]:
# ── Création du label ─────────────────────────────────────────────────────
SEUIL_GROS = 1000
rfm['label'] = (rfm['Monetary'] > SEUIL_GROS).astype(int)

print(f"📊 Distribution des classes (seuil = {SEUIL_GROS} €) :")
vc = rfm['label'].value_counts()
for k, v in vc.items():
    label = "Gros dépensier (1)" if k == 1 else "Petit dépensier (0)"
    print(f"   {label} : {v:,} ({v/len(rfm)*100:.1f}%)")

# ── Features & split ────────────────────────────────────────────────────
X = rfm[['Recency', 'Frequency']]
y = rfm['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print(f"\n✅ Train : {len(X_train):,} | Test : {len(X_test):,}")


### 🟢 5.3 Entraînement du Random Forest baseline

In [ ]:
# ── Modèle baseline ───────────────────────────────────────────────────────
rf_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)

y_pred_base = rf_base.predict(X_test)
y_proba_base = rf_base.predict_proba(X_test)[:, 1]

print("=" * 50)
print("  RÉSULTATS — Random Forest Baseline")
print("=" * 50)
print(classification_report(y_test, y_pred_base,
      target_names=['Petit dépensier', 'Gros dépensier']))
print(f"  AUC-ROC  : {roc_auc_score(y_test, y_proba_base):.4f}")

# ── Cross-validation 5-fold ────────────────────────────────────────────
cv_scores = cross_val_score(rf_base, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                              scoring='f1', n_jobs=-1)
print(f"\n📊 Cross-validation F1 (5-fold) : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


---
## 6️⃣ Optimisation GridSearchCV & Évaluation Finale

### 🔵 6.1 PySpark — Résultats optimisés (code fourni section 5.1)


### 🟢 6.2 Pandas — GridSearchCV

In [ ]:
# ── Grid Search ────────────────────────────────────────────────────────
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth':    [5, 10, None],
    'min_samples_leaf': [1, 5],
}

gs = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=-1,
    verbose=0
)

print("⏳ GridSearch en cours...")
gs.fit(X_train, y_train)

print(f"\n🏆 Meilleurs hyperparamètres : {gs.best_params_}")
print(f"   Meilleur F1 (CV) : {gs.best_score_:.4f}")


In [ ]:
# ── Évaluation du meilleur modèle ────────────────────────────────────────
best_rf = gs.best_estimator_
y_pred_best  = best_rf.predict(X_test)
y_proba_best = best_rf.predict_proba(X_test)[:, 1]

print("=" * 50)
print("  RÉSULTATS — Meilleur Modèle (GridSearchCV)")
print("=" * 50)
print(classification_report(y_test, y_pred_best,
      target_names=['Petit dépensier', 'Gros dépensier']))
print(f"  AUC-ROC  : {roc_auc_score(y_test, y_proba_best):.4f}")
print()

# Comparaison baseline vs optimisé
print("📊 Comparaison Baseline vs Optimisé :")
print(f"   F1 Baseline   : {roc_auc_score(y_test, y_proba_base):.4f}")
print(f"   F1 Optimisé   : {roc_auc_score(y_test, y_proba_best):.4f}")


In [ ]:
# ── Visualisations finales : Confusion Matrix + ROC Curve ────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("📈 Évaluation Finale du Modèle", fontsize=15, fontweight='bold')

# 1. Matrice de confusion
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Petit (0)', 'Gros (1)'],
            yticklabels=['Petit (0)', 'Gros (1)'],
            linewidths=1, linecolor='white', cbar=False, annot_kws={'size': 14})
axes[0].set_xlabel('Prédiction', fontweight='bold')
axes[0].set_ylabel('Réalité', fontweight='bold')
axes[0].set_title('Matrice de Confusion', fontweight='bold')

# 2. Courbe ROC
fpr_b, tpr_b, _ = roc_curve(y_test, y_proba_base)
fpr_o, tpr_o, _ = roc_curve(y_test, y_proba_best)
auc_b = roc_auc_score(y_test, y_proba_base)
auc_o = roc_auc_score(y_test, y_proba_best)

axes[1].plot(fpr_b, tpr_b, color=PALETTE[1], lw=2,
             label=f'Baseline (AUC = {auc_b:.3f})', linestyle='--')
axes[1].plot(fpr_o, tpr_o, color=PALETTE[0], lw=2.5,
             label=f'Optimisé (AUC = {auc_o:.3f})')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.3, lw=1)
axes[1].fill_between(fpr_o, tpr_o, alpha=0.08, color=PALETTE[0])
axes[1].set_xlabel('Taux Faux Positifs', fontweight='bold')
axes[1].set_ylabel('Taux Vrais Positifs', fontweight='bold')
axes[1].set_title('Courbe ROC', fontweight='bold')
axes[1].legend(fontsize=10)

# 3. Importance des features
importances = pd.Series(best_rf.feature_importances_,
                         index=['Recency', 'Frequency'])
colors_imp = [PALETTE[0] if v == importances.max() else PALETTE[1]
              for v in importances.values]
bars = axes[2].barh(importances.index, importances.values,
                    color=colors_imp, edgecolor='white', height=0.4)
axes[2].set_title('Importance des Variables', fontweight='bold')
axes[2].set_xlabel('Importance (Gini)')
for bar, val in zip(bars, importances.values):
    axes[2].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()


---
## 🏁 Conclusion & Recommandations Stratégiques

### Récapitulatif du Pipeline


In [ ]:
print("=" * 60)
print("  BILAN DU PROJET — SEGMENTATION E-COMMERCE")
print("=" * 60)

print(f"\n📊 Dataset analysé :")
print(f"   • {df.shape[0]:,} transactions | {df_clean['CustomerID'].nunique():,} clients | 38 pays")

print(f"\n🔵 PySpark (Big Data) :")
print("   • Pipeline distribué sur 541 909 transactions")
print("   • BisectingKMeans + Silhouette pour sélection du k optimal")
print("   • CrossValidator 5-fold pour l'optimisation supervisée")

print(f"\n🟢 Scikit-learn (Machine Learning) :")
seg_summary_final = rfm.groupby('Segment').agg(
    n=('CustomerID','count'), m=('Monetary','mean')).sort_values('m', ascending=False)
for seg, row in seg_summary_final.iterrows():
    icon = rfm[rfm['Segment']==seg]['SegIcon'].iloc[0]
    print(f"   {icon} {seg:<30} : {int(row['n']):>4} clients | Montant moy. {row['m']:>8,.0f} €")

print(f"\n🤖 Modèle prédictif (Random Forest optimisé) :")
print(f"   • AUC-ROC     : {roc_auc_score(y_test, y_proba_best):.4f}")
print(f"   • F1-Score    : {gs.best_score_:.4f}")
print(f"   • Paramètres  : {gs.best_params_}")

print("\n💡 Recommandations :")
print("   ⭐ Champions VIP      → Fidélisation Platine + parrainage")
print("   💚 Fidèles Réguliers  → Upsell cross-sell + programme points")
print("   🌱 Clients Potentiel  → Nurturing + offres de montée en gamme")
print("   ⚠️  À Risque           → Campagne Win-back + code promo -20%")
